## Calculating a Typical Meteorological Year

This notebook walks through the process of calculating a [Typical Meteorological Year](https://nsrdb.nrel.gov/data-sets/tmy), an hourly dataset used for applications in energy and building systems modeling. Because this represents average rather than extreme conditions, an TMY dataset is not suited for designing systems to meet the worst-case conditions occurring at a location. 

The TMY methodology here mirrors that of the Sandia/NREL TMY3 methodology, and uses historic and projected downscaled climate data available through the Cal-Adapt: Analytics Engine catalog. As this methodology heavily weights the solar radiation input data, be aware that the final selection of "typical" months may not be typical for other variables. 

Please note, the Analytics Engine also has an *Average Meteorological Year* functionality. The methods shown throughout this notebook will soon replace the underlying backend `climakitae` code for the AMY in order to better address our user needs, i.e., we are working to replace the AMY with the TMY methods. We provide this walkthrough to demonstrate confidence in the "AMY to TMY" conversion process for our users in the meantime. 

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import climakitae as ck
import pandas as pd
import xarray as xr
import numpy as np
import pkg_resources
import warnings 
warnings.filterwarnings("ignore")

from climakitae.utils import get_closest_gridcell

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

### Step 1: Grab and process all required input data

The [TMY3 method](https://www.nrel.gov/docs/fy08osti/43156.pdf) selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance. Some of these variables are already available in the Analytics Engine data catalog, and some we will need to calculate ourselves.  

#### Step 1a: Select location of interest
TMYs are usually calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the TMY.  In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too. 

In [ ]:
# read in station file of CA HadISD stations
stn_file = pkg_resources.resource_filename("climakitae", "data/hadisd_stations.csv")
stn_file = pd.read_csv(stn_file, index_col=[0])
stn_file.head(5)

In [ ]:
# grab los angeles airport
station_name = "Los Angeles International Airport"
one_station = stn_file.loc[stn_file['station'] == station_name]
stn_lat = one_station.LAT_Y.values
stn_lon = one_station.LON_X.values
stn_lat, stn_lon

Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol.

In [ ]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# station_name = 'YOUR_STATION_NAME_HERE'

#### Step 1b: Variables in catalog
When selecting data, there are several considerations to be aware of. The recommended minimum input of data is 15-20 years worth of daily data. First, we will pre-load some data options to ensure that the same constraints are applied to every variable we retrieve from the catalog and calculate. For example, we will process the data for our designated station location (latitude, and longitude) at 45 km over the 1990-2020 period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

In [ ]:
# default selections applicable to all variables selected
app.selections.data_type = "Gridded"
app.selections.area_average = "No"
app.selections.scenario_historical = ["Historical Climate"]
app.selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"] # Important based on time period considered!!
app.selections.time_slice = (1990, 2020)
app.selections.timescale = "daily"
app.selections.resolution = "45 km"
app.selections.cached_area = 'coordinate selection'
app.selections.latitude = (stn_lat-1., stn_lat+1.)
app.selections.longitude = (stn_lon-1., stn_lon+1.)

Now that we have set up default settings, let's start retrieving data! First, retrieve min, max, and mean air temperature: 

In [ ]:
# max air temp
app.selections.variable = "Maximum air temperature at 2m"
app.selections.unit = "degC"
max_airtemp_data = app.retrieve()
max_airtemp_at_stn = get_closest_gridcell(max_airtemp_data, stn_lat, stn_lon) # retrieves data at lat-lon location

# min air temp
app.selections.variable = "Minimum air temperature at 2m"
app.selections.unit = "degC"
min_airtemp_data = app.retrieve()
min_airtemp_at_stn = get_closest_gridcell(min_airtemp_data, stn_lat, stn_lon) # retrieves data at lat-lon location

# mean air temperature
app.selections.variable = "Air Temperature at 2m" 
app.selections.unit = "degC"
mean_airtemp_data = app.retrieve()
mean_airtemp_data.name = "Mean air temperature at 2m" # rename for clarity
mean_airtemp_at_stn = get_closest_gridcell(mean_airtemp_data, stn_lat, stn_lon) # retrieves data at lat-lon location

Retrieve max and mean wind speed:

In [ ]:
# max wind speed
app.selections.variable = "Maximum wind speed at 10m"
app.selections.unit = "m s-1"
max_windspd_data = app.retrieve()
max_windspd_at_stn = get_closest_gridcell(max_windspd_data, stn_lat, stn_lon) # retrieves data at lat-lon location

# mean wind speed
app.selections.variable = "Mean wind speed at 10m"
app.selections.unit = "m s-1"
mean_windspd_data = app.retrieve()
mean_windspd_at_stn = get_closest_gridcell(mean_windspd_data, stn_lat, stn_lon) # retrieves data at lat-lon location

#### Step 1c: Variables that need to be calculated
Next, we will need to retrieve **hourly** data from the catalog to calculate the following variables, as they are not natively within the Analytics Engine data catalog. 
- Max, min, and mean dew point temperature
- Global horizontal irradiance
- Direct normal irradiance **coming to the AE catalog soon** (code is provided to calculate it, but it is unfinished)

In [ ]:
# switch to hourly timescale
app.selections.timescale = "hourly"

Next, calculate max, min, and mean dew point temperature:

In [ ]:
# dew point temperature
app.selections.variable = "Dew point temperature"
app.selections.unit = "degC"
dewpt_data = app.retrieve()
dewpt_data_at_stn = get_closest_gridcell(dewpt_data, stn_lat, stn_lon) # retrieves data at lat-lon location

In [ ]:
max_dewpt_data_at_stn = dewpt_data_at_stn.resample(time="1D").max() # daily max dewpoint temp
max_dewpt_data_at_stn.name = "Daily max dewpoint temperature" # rename for clarity

min_dewpt_data_at_stn = dewpt_data_at_stn.resample(time="1D").min() # daily min dewpoint temp
min_dewpt_data_at_stn.name = "Daily min dewpoint temperature" # rename for clarity

mean_dewpt_data_at_stn = dewpt_data_at_stn.resample(time="1D").mean() # daily mean dewpoint temp
mean_dewpt_data_at_stn.name = "Daily mean dewpoint temperature" # rename for clarity

Next, retrieve global horizontal irradiance. GHI is within the Analytics Enginge catalog at daily resolutions, but for the TMY methodology, we need to calculate the total accumulated GHI received over the course of the day, so we will retrieve hourly data instead and calculate the necessary information below. The same process is repeated for the direct normal irradiance (once the data is in the AE catalog). 

In [ ]:
# global irradiance
app.selections.variable = "Instantaneous downwelling shortwave flux at bottom"
global_irradiance_data = app.retrieve()
global_irradiance_data.name = "Global horizontal irradiance" # rename for clarity
ghi_at_stn = get_closest_gridcell(global_irradiance_data, stn_lat, stn_lon) # retrieves data at lat-lon location
total_ghi_at_stn = ghi_at_stn.resample(time="1D").sum() # total global horizontal irradiance (accumulation of hourly data over a 24-hour period)

In [ ]:
## ONCE IN CATALOG (AND REMOVE DNI CALCULATION BELOW)

# # direct normal irradiance
# app.selections.variable = "VARIABLE_NAME_HERE"
# direct_irradiance_data = app.retrieve()
# direct_irradiance_data.name = "Direct normal irradiance" # rename for clarity
# dni_at_stn = get_closest_gridcell(direct_irradiance_data, stn_lat, stn_lon) # retrieves data at lat-lon location
# total_dni_at_stn = dni_at_stn.resample(time="1D").sum() # total direct normal irradiance (accumulation of hourly data over a 24-hour period)

---
NOTE: DELETE THESE CELLS ONCE DNI IN CATALOG. THIS IS SEMI-CLOSE TO WHAT WE NEED, BUT MORE EFFICIENT TO JUST WAIT UNTIL DNI IN CATALOG

Lastly, we will calculate **direct normal irradiance**. As direct normal irradiance is not presently in the Analytics Engine catalog (but is coming soon!), we will use the DIRINT method with the dewpoint temperature enhancement ([Perez et al. 1992](https://www.researchgate.net/publication/279868352_Dynamic_global-to-direct_irradiance_conversion_models)) to calculate, which uses the **global horizontal irradiance** and the solar zenith angle. 

Once direct normal irradiance is in the Analytics Engine data catalog, we will update this notebook to reflect the changes as well. 

**notes**
- *Perez, R., P. Ineichen, E. Maxwell, R. Seals and A. Zelenka, (1992). “Dynamic Global-to-Direct Irradiance Conversion Models”. ASHRAE Transactions-Research Series, pp. 354-369*
- *Different methods: https://assessingsolar.org/notebooks/decomposition_models.html*

First calculate the solar zenith angle: 

In [ ]:
# # calculate solar position, including solar zenith angle
# solpos = solarposition.get_solarposition(time = pd.date_range('1990-01-01', '2021-01-01', freq='H'), # need to avoid hardcoding dates here... in progress
#                                         latitude = stn_lat,
#                                         longitude = stn_lat)
# solpos

Next, we will set-up a helper function `dni_by_sim` to calculate direct normal irradiance. 

In [ ]:
# def dni_by_sim(ghi_data, dew_data, solpos):
#     """Calculates the direct normal irradiance per simulation, with DIRINT method and dew point improvements"""
#     df = pd.DataFrame()
#     for sim in range(4):
#         df_by_sim = ghi_data.squeeze().isel(simulation=sim).to_dataframe()
#         dt_by_sim = dew_data.squeeze().isel(simulation=sim).to_dataframe()
#         dirint_by_sim = pvlib.irradiance.dirint(ghi=df_by_sim['Global horizontal irradiance'],
#                                                 temp_dew=dt_by_sim['Dew point temperature'],
#                                                 solar_zenith=solpos['apparent_zenith'],
#                                                 times=solpos.index)
        
#         df[df_by_sim.simulation.values[0]] = dirint_by_sim      
#     return df

Calculate direct normal irradiance and check out the values next:

In [ ]:
# # need to adjust for differences in calendar (leap days)

# dni = dni_by_sim(ghi_at_stn, dewpt_data, solpos) # hourly direct normal irradiance
# total_dni_at_stn = dni.resample(time="1D").sum() # total direct normal irradiance (accumulation of hourly data over a 24-hour period)
# total_dni_at_stn

---

#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now. 

In [ ]:
all_vars = xr.merge([max_airtemp_at_stn.squeeze(), min_airtemp_at_stn.squeeze(), mean_airtemp_at_stn.squeeze(),
                     max_dewpt_data_at_stn.squeeze(), min_dewpt_data_at_stn.squeeze(), mean_dewpt_data_at_stn.squeeze(),
                     max_windspd_at_stn.squeeze(), mean_windspd_at_stn.squeeze(),
                     total_ghi_at_stn.squeeze()]) #, total_dni_at_stn]) # will uncomment this once DNI in catalog

all_vars

In [ ]:
# load all indices in
all_vars = all_vars.compute()

In [ ]:
all_vars

### Step 2: Calculate cumultative distribution functions

For the TMY, the cumulative distribution function gives the proportion of values that are less than or equal to a specified value of the index. In this case, we want to identify months that are as close to the long-term climatology for each variable as possible, indicating months that are "typical".  

#### Step 2a: Calculate long-term climatology CDFs for each index
First, we need to calculate the long-term climatology for each index for each month so we can establish the baseline pattern. 

In [ ]:
def compute_cdf(da): 
    """Compute the cumulative density function for an input DataArray"""
    da_np = da.values # Get numpy array of values
    num_samples = 1024 # Number of samples to generate
    count, bins_count = np.histogram( # Create a numpy histogram of the values 
        da_np, bins = np.linspace(
            da_np.min(), # Start at the minimum value of the array 
            da_np.max(), # End at the maximum value of the array 
            num_samples
        )
    )
    cdf_np = np.cumsum(count/sum(count)) # Compute the CDF 
    
    # Turn the CDF array into xarray DataArray 
    # New dimension is the bin values 
    cdf_da = xr.DataArray( 
        [bins_count[1:],cdf_np],
        dims=["data","bin_number"],
        coords = {
            "data":["bins","probability"],
        }
    )
    cdf_da.name = da.name
    return cdf_da

def get_cdf_by_sim(da): 
    # Group the DataArray by simulation
    return da.groupby("simulation").apply(compute_cdf)

def get_cdf_by_mon_and_sim(da): 
    # Group the DataArray by month in the year 
    return da.groupby("time.month").apply(get_cdf_by_sim)

def get_cdf(ds): 
    """Get the cumulative density function. 
    
    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """
    return ds.apply(get_cdf_by_mon_and_sim)

In [ ]:
cdf_climatology = get_cdf(all_vars)

Next, we set up a handy plotting function so that we can view the CDF patterns for each variable. 

In [ ]:
def plot_one_var_cdf(cdf_da): 
    """Plot CDF for a single variable 
    Written to function for the unique configuration of the CDF DataArray object
    Silences an annoying hvplot warning 
    Will show every simulation together on the plot 
    
    Parameters
    -----------
    cdf: xr.DataArray 
       Cumulative density function for a single variable 
    
    Returns 
    -------
    panel.layout.base.Column
        Hvplot lineplot 
        
    """
    prob_da = cdf_da.sel(data="probability",drop=True).rename("probability") # Grab only probability da 
    bins_da = cdf_da.sel(data="bins",drop=True).rename("bins") # Grab just bin values
    ds = xr.merge([prob_da,bins_da]) # Merge the two to form a single Dataset object
    cdf_pl = ds.hvplot(
        "bins","probability", 
        by="simulation", # Simulations should all be displayed together 
        widget_location="bottom",
        grid=True,
        xlabel="{0} ({1})".format(var,cdf_da.attrs["units"]), 
        xlim=(bins_da.min().item(), bins_da.max().item()), # Fix the x-limits for all months
        ylabel="Probability (0-1)", 
    )
    return cdf_pl 

In the plot below, we'll display maximum air temperature to assess the climatological CDF pattern, but you can modify the variable here to one of your choosing to see the pattern too! Also select a different month by moving the slider bar to see the pattern throughout the year. 

In [ ]:
# Choose your desired variable
var = "Maximum air temperature at 2m"

# Make the plot
cdf_plot = plot_one_var_cdf(cdf_climatology[var])
display(cdf_plot)

#### Step 2b: Calculate CDFs for each index for all months

Next, we will calculate CDF for each month and each variable, for which we ultimatley will compare against climatology. For the individual months, we must also exclude the period of time during a major volcanic eruption (Pinatubo: June 1991 to December 1994) as the aerosols have an impact on solar variables. The cells below functions exlude this data from our data next. 

In [ ]:
def get_cdf_monthly(ds):
    """Get the cumulative density function by unique mon-yr combos
    
    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """
    def get_cdf_mon_yr(da): 
        return da.groupby("time.year").apply(get_cdf_by_mon_and_sim)
    return ds.apply(get_cdf_mon_yr)

In [ ]:
cdf_monthly = get_cdf_monthly(all_vars)
cdf_monthly

Now we'll remove the volcanic years: 

In [ ]:
# Remove the years for the Pinatubo eruption 
cdf_monthly = cdf_monthly.where(
    (~cdf_monthly.year.isin([1991,1992,1993,1994])), 
    np.nan, drop=True)

Like the climatology CDF figure above, let's check out the individual months next. You can modify the variable, and month-year to display too. 

In [ ]:
# Choose your desired variable
var = "Minimum air temperature at 2m"

# Make the plot
cdf_plot_mon_yr = plot_one_var_cdf(cdf_monthly[var])
display(cdf_plot_mon_yr)

### Step 3: Compare climatology CDF to monthly CDF for each variable

Now that we hvae the distributions for the long-term climatology of our 30-year period, and the corresponding distribution for each month in that 30-year period, we need to assess how each individual month compares to the long-term climatology. In essence, we are looking for the individual months that best capture the climatology distribution. First, we will need to calculate the Finkelstein-Schafer statistic which assesses how close the distributions are to each other. 

#### Step 3a: Calculate the Finkelstein-Schafer statistic 
The [Finkelstein-Schafer statistic](https://academic.oup.com/biomet/article-abstract/58/3/641/233677) determines the absolute difference between the long-term climatology and candidate CDF profiles, and considers the number of days within each month. We will use a handy function `fs_statistic` to calculate this below. 

In [ ]:
def fs_statistic(cdf_climatology, cdf_month):
    """
    Calculates the Finkelstein-Schafer statistic:
    Absolute difference between long-term climatology and candidate CDF, divided by number of days in month
    """
    days_per_mon = xr.DataArray(
        data=[31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], 
        coords={"month":np.arange(1,13)}
    )
    fs_stat = abs(cdf_monthly - cdf_climatology).sel(data="probability") / days_per_mon
    return fs_stat

In [ ]:
all_vars_fs = fs_statistic(cdf_climatology, cdf_monthly)

#### Step 3b: Weight the F-S statistic

Next, we weight the F-S statistic results based on the input variables. The [TMY3](https://www.nrel.gov/docs/fy08osti/43156.pdf) method places a higher weight to the solar variables (global irradiance and direct irradiance), which we follow here. 

In [ ]:
def compute_weighted_fs(da_fs):
    """Weights the F-S statistics based on TMY3 methodology"""
    weights_per_var = {
        'Maximum air temperature at 2m':1/20,
        'Minimum air temperature at 2m':1/20,
        'Mean air temperature at 2m':2/20,
        'Daily max dewpoint temperature':1/20,
        'Daily min dewpoint temperature':1/20,
        'Daily mean dewpoint temperature':2/20,
        'Maximum wind speed at 10m':1/20,
        'Mean wind speed at 10m':1/20,
        'Global horizontal irradiance':5/20
        # 'Direct normal irradiance':5/20
    }

    for var, weight in weights_per_var.items(): 
        # Multiply each variable by it's appropriate weight 
        da_fs[var] = da_fs[var]*weight 
    return da_fs

In [ ]:
weighted_fs = compute_weighted_fs(all_vars_fs)
weighted_fs

#### Step 3c: Select candidate months for consideration
Once weighted, we select the top 5 candidate months for each month that have lowest weighted sums, meaning that these candidate months are the closest to the long-term climatology for that month. 

In [ ]:
# Sum 
weighted_fs_sum = weighted_fs.to_array().sum(dim=["variable","bin_number"])

# Stack by simulation and year
stacked_fs = weighted_fs_sum.stack(stacked=["simulation","year"])

# Get top 5 candidate months by per month
# Print year and simulation corresponding to the top 5 candidate months 
da_list = []
for mon in np.arange(1,12+1): 
    top_5_by_mon = stacked_fs.sel(month=mon).sortby(stacked_fs.sel(month=mon), ascending=True)[:5].expand_dims("month")
    print("month: {0}".format(mon))
    print(top_5_by_mon.year.values)
    print(top_5_by_mon.simulation.values)
    da_list.append(top_5_by_mon)

#### Step 3d: Determine the TMY months

Now, let's select the representative month for each month of the year!Now, let's select the representative month for each month of the year.

In [ ]:
tmy_month_list = []
for mon in np.arange(0,12):
    tmy_month_list.append(str(da_list[mon][:,0].year.values) + "-" + '{:02d}'.format(mon+1))

In [ ]:
tmy_month_list

The list above represents the ideal months that represent the long term climatology based on the 10 indices. Meaning, for a "typical" meteorological year, data for Jan will come from Jan 2011, data for Feb will come from 1996, and so on. In the next step, we will generate the resulting "TMY" file that is commonly used in such applications. 

### Step 4: Generate the TMY data outputs

Generally, the following data is outputted using the TMY months:
- Date & time (UTC)
- Air temperate at 2m [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2] -- ***coming soon to AE***
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [ ]:
# this function should be cleaned up a bit, but is functional for what we need

def generate_tmy_data(tmy_month_list):
    vars_to_pull = ['Air Temperature at 2m', # degC
                    'Relative humidity', # % 
                    'Instantaneous downwelling shortwave flux at bottom', # W m-2
                    # direct irradiance here, ## once in catalog # W m-2
                    'Shortwave surface downward diffuse irradiance', # W m-2
                    'Instantaneous downwelling longwave flux at bottom', # W m-2
                    'Wind speed at 10m', # m s-1
                    'Wind direction at 10m', # degrees
                    'Surface Pressure'] # Pa
        
    # Set up dataframe for easy export as csv
    tmy_data_to_export = pd.DataFrame()
    
    # Set up catalog access settings
    app.selections.data_type = "Gridded"
    app.selections.area_average = "No"
    app.selections.scenario_historical = ["Historical Climate"]
    app.selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"] # Important based on time period considered!!
    app.selections.timescale = "hourly"
    app.selections.resolution = "45 km"
    app.selections.cached_area = 'coordinate selection'
    app.selections.latitude = (stn_lat-1., stn_lat+1.)
    app.selections.longitude = (stn_lon-1., stn_lon+1.)
    
    # Grab data per variable and add to dataframe
    for month in tmy_month_list:
        for var in vars_to_pull:
            print('Retrieving {0} data for: {1}'.format(month, var))
            
            # Air temperature is only var with non-default unit
            if var == "Air Temperature at 2m":
                app.selections.units = "degC"
                app.selections.variable = var
                app.selections.time_slice = (int(month[:4]), int(month[:4])) # retrieves the year of the TMY month

                data = app.retrieve() # retrieves data selections

                # Selects data at lat-lon location, turning off print_coords statement to clear output
                data_at_stn = get_closest_gridcell(data, stn_lat, stn_lon, print_coords=False)

                # Selects data from the tmy designated month
                monthly_data = data_at_stn.squeeze().sel(time=month).drop(['lakemask','landmask','x','y','Lambert_Conformal']).to_dataframe()
                
                # Concatenate months together
                tmy_data_to_export = pd.concat([tmy_data_to_export, monthly_data], axis=0) # Concatenate months together

            else:
                app.selections.variable = var
                app.selections.time_slice = (int(month[:4]), int(month[:4])) # retrieves the year of the TMY month

                data = app.retrieve() # retrieves data selections

                # Selects data at lat-lon location, turning off print_coords statement to clear output
                data_at_stn = get_closest_gridcell(data, stn_lat, stn_lon, print_coords=False)

                # Selects data from the tmy designated month
                monthly_data_var = data_at_stn.squeeze().sel(time=month).drop(['lakemask','landmask','x','y','Lambert_Conformal']).to_dataframe()

            # Add each variable within a month as a new column
            if var != vars_to_pull[0]:
                monthly_data[var] = monthly_data_var[var].values
            
            # Concatenate months together
            tmy_data_to_export = pd.concat([tmy_data_to_export, monthly_data], axis=0) 
    
    return tmy_data_to_export

In the next cell, we will run the `generate_tmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take quite a bit of time! On the Analytics Engine JupyterHub, this step takes approximately 20 minutes. 

In [ ]:
tmy_data_to_export = generate_tmy_data(tmy_month_list)
tmy_data_to_export

Lastly, let's export the TMY data below as a csv file. 

In [ ]:
filename = 'TMY_{}'.format(format(station_name.replace(" ", "_")).lower())
tmy_data_to_export.to_csv(filename + '.csv')